# RLN Injury Sensitivity Analysis v2

Compare CLN-RLN association OR across tier definitions:
- **Tier 1 only**: Laryngoscopy-confirmed (6 patients)
- **Tier 1+2**: + Chart-documented (25 patients)
- **Refined Tier 3**: Context-filtered NLP (92 patients total)
- **Original (all tiers)**: Unfiltered NLP (679 patients)

Uses `extracted_rln_injury_refined_v2` and `vw_patient_postop_rln_injury_detail`.

In [ ]:
import toml
import duckdb
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

secrets = toml.load('../.streamlit/secrets.toml')
token = secrets['MOTHERDUCK_TOKEN']
con = duckdb.connect(f'md:thyroid_research_2026?motherduck_token={token}')
print('Connected to MotherDuck')

In [ ]:
# Build CLN exposure variable
cln_sql = """
WITH surgical AS (
    SELECT CAST(research_id AS INT) AS research_id,
           LOWER(COALESCE(thyroid_procedure, '')) AS proc,
           CASE WHEN central_compartment_dissection IS NOT NULL
                  OR LOWER(COALESCE(tumor_1_level_examined, '')) LIKE '%6%'
                  OR LOWER(COALESCE(other_ln_dissection, '')) LIKE '%central%'
                  OR LOWER(COALESCE(other_ln_dissection, '')) LIKE '%level 6%'
                  OR LOWER(COALESCE(tumor_1_ln_location, '')) LIKE '%perithyroidal%'
                  OR LOWER(COALESCE(tumor_1_ln_location, '')) LIKE '%pretracheal%'
                  OR LOWER(COALESCE(tumor_1_ln_location, '')) LIKE '%paratracheal%'
                  OR LOWER(COALESCE(tumor_1_ln_location, '')) LIKE '%delphian%'
                  OR LOWER(COALESCE(tumor_1_ln_location, '')) LIKE '%prelaryngeal%'
           THEN 1 ELSE 0 END AS cln_flag
    FROM path_synoptics
    WHERE TRY_CAST(surg_date AS DATE) IS NOT NULL
)
SELECT research_id, MAX(cln_flag) AS has_cln
FROM surgical
GROUP BY research_id
"""
df_cln = con.execute(cln_sql).fetchdf()
print(f'CLN cohort: {len(df_cln)} patients, {df_cln["has_cln"].sum()} with CLN')

In [ ]:
# Get RLN injury flags for each tier definition

# Original (all tiers, unfiltered)
df_orig = con.execute("""
    SELECT DISTINCT research_id, 1 AS rln_original
    FROM vw_patient_postop_rln_injury_detail
""").fetchdf()

# Tier 1 only
df_t1 = con.execute("""
    SELECT DISTINCT research_id, 1 AS rln_tier1
    FROM vw_patient_postop_rln_injury_detail
    WHERE source_tier = 'laryngoscopy_confirmed'
""").fetchdf()

# Tier 1+2
df_t12 = con.execute("""
    SELECT DISTINCT research_id, 1 AS rln_tier12
    FROM vw_patient_postop_rln_injury_detail
    WHERE source_tier IN ('laryngoscopy_confirmed', 'chart_documented')
""").fetchdf()

# Refined (new view)
df_refined = con.execute("""
    SELECT DISTINCT research_id, 1 AS rln_refined
    FROM extracted_rln_injury_refined_v2
""").fetchdf()

# Refined confirmed only
df_confirmed = con.execute("""
    SELECT DISTINCT research_id, 1 AS rln_confirmed
    FROM extracted_rln_injury_refined_v2
    WHERE rln_injury_is_confirmed = TRUE
""").fetchdf()

print(f'Original: {len(df_orig)}, Tier1: {len(df_t1)}, Tier1+2: {len(df_t12)}, '
      f'Refined: {len(df_refined)}, Confirmed: {len(df_confirmed)}')

In [ ]:
# Merge all
df = df_cln.copy()
for label, df_rln in [('rln_original', df_orig), ('rln_tier1', df_t1),
                       ('rln_tier12', df_t12), ('rln_refined', df_refined),
                       ('rln_confirmed', df_confirmed)]:
    df = df.merge(df_rln, on='research_id', how='left')
    df[label] = df[label].fillna(0).astype(int)

print(f'Merged cohort: {len(df)} patients')
df.describe()

In [ ]:
# Fisher exact test for each tier definition
results = []
for label in ['rln_original', 'rln_tier1', 'rln_tier12', 'rln_refined', 'rln_confirmed']:
    ct = pd.crosstab(df['has_cln'], df[label])
    # Ensure 2x2
    for c in [0, 1]:
        if c not in ct.columns:
            ct[c] = 0
    for r in [0, 1]:
        if r not in ct.index:
            ct.loc[r] = 0
    ct = ct.sort_index().sort_index(axis=1)
    table = [[ct.loc[1,1], ct.loc[1,0]], [ct.loc[0,1], ct.loc[0,0]]]
    oddsratio, pvalue = fisher_exact(table)
    n_rln = df[label].sum()
    rate = 100 * n_rln / len(df)
    results.append({
        'Definition': label.replace('rln_', ''),
        'N_RLN': int(n_rln),
        'Rate_%': round(rate, 2),
        'OR (CLN vs no-CLN)': round(oddsratio, 3),
        'p-value': f'{pvalue:.4f}',
    })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

In [ ]:
# Forest plot of ORs
fig, ax = plt.subplots(figsize=(8, 4))
labels = df_results['Definition'].values
ors = df_results['OR (CLN vs no-CLN)'].values
ns = df_results['N_RLN'].values

y_pos = range(len(labels))
colors = ['#d62728', '#2ca02c', '#2ca02c', '#1f77b4', '#1f77b4']

ax.barh(y_pos, ors, color=colors, height=0.6, alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels([f'{l} (n={n})' for l, n in zip(labels, ns)])
ax.axvline(1.0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Odds Ratio (CLN vs no-CLN)')
ax.set_title('CLN-RLN Association by Tier Definition')

for i, (o, p) in enumerate(zip(ors, df_results['p-value'].values)):
    ax.text(o + 0.02, i, f'OR={o:.2f}, p={p}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../studies/hypothesis1_cln_lobectomy/rln_sensitivity_forest.png', dpi=150)
plt.show()
print('Saved forest plot')

In [ ]:
# Tier breakdown of refined view
tier_summary = con.execute("""
    SELECT * FROM extracted_rln_injury_refined_summary_v2
""").fetchdf()
print('=== Refined Summary ===')
print(tier_summary.to_string(index=False))

exclusion = con.execute("""
    SELECT * FROM extracted_rln_exclusion_audit_v2
""").fetchdf()
print('\n=== Exclusion Audit ===')
print(exclusion.to_string(index=False))

In [ ]:
# Save results
df_results.to_csv('../exports/rln_sensitivity_analysis_v2.csv', index=False)
print('Saved to exports/rln_sensitivity_analysis_v2.csv')
con.close()